## Road Surface Classification Using Smartphone Accelerometer Data

## Team Members
- Samuel Cramer

## GitHub Repository URL
https://github.com/scramer27/road_surface_classification/

## Abstract

Potholes are a problem for both cities and drivers, causing billions of dollars of damage to vehicles every year. While just repairing potholes is relatively inexpensive, identifying their locations is currently very costly. Manually evaluating road surfaces is a lengthy, labor-intensive, process, and some using modern techniques such as using LIDAR and advanced cameras can be rather expensive. In this project, we have developed machine learning models in order to help detect potholes and normal road surfaces using smartphone accelerometer data. We compared baseline methods, logistic regression models, and a 1D convolutional neural network (CNN) trained on time-series accelerometer windows, with labels indicating whether each window corresponds to a pothole or normal surface. Our results show that model complexity improved pothole detection performance, as engineered features in a logistic regression model and CNN methods significantly improved our metrics when compared to simple baselines or a logistic regression model on raw data.

## Introduction

Maintaining high-quality road infrastucture is an essential aspect of municipal operations and public safety. Potholes, however, are road defects can form from many causes such as moisture, traffic, and poor installation. However, perhaps the most relevant to my current residence, Vermont, would be the freeze-thaw damages that is made especially apparent in the mud season from April to May. Depending on severity, damage caused by potholes can range from a small bump to bent wheels to event multi-vehicle accidents. In fact, potholes and other similar road defects in the United States can cost drivers upwards of $3 billion in annual repairs. While potholes can be cost effective to repair, at roughly $200 each, detecting potholes remains difficult and costly. Many municipalities still rely on manual reporting or inspection, while other options such as Vermont's Automated Road Analyzer (ARAN) make use of expensive lazer technology. 

Smartphone-based detection, however, may offer a great alternative. While few workers are tasked with road maintenance, nearly every driver carries a device with an accelerometer and GPS. There have been many others who have experimented within this realm. For one, Eriksson et al. (2008) created the pothole patrol system by utilizing a collection of vehicles equipped with 3-axis accelerometers and GPS, in order to map high-confidence road anomaly detections, using clustering techniques. Furthermore, Kulkarni et al. (2014) utilized accelerometers and GPS on an Android operating system, along with a neural network to achieve high performance (over 90% accuracy), setting effective acceleration threshold values on the x-axis and z-axis. Finally, Mednis et al. (2011) were able to tune a Z-axis acceleration algorithm under real-world driving conditions, with TP rates up to 92%. Our work attempts to build on the aforementioned, collecting our own labeled dataset, performing preprocessing, and then comparing four levels of model complexity on the same dataset, using a 1D CNN as our most advanced machine learning technique.

Being able to perform automatic road surface classification is extremely beneficial. By detecting and geolocating potholes automatically, all road defects of a road network can be identified more easily. If road workers were able to have this at their disposal, they would be able to mindlessly drive over road surfaces while their phone captures the geolocations of imperfections and normal roads alike, while if "crowd-sourced" by a city's residents, the quantity of data would speed up the detection process and lessen the strain on city workers and budgets. 

Note: References are located at the bottom of the notebook

## Values Statement
The primary users of this project would be governments, municipalities, individuals, and contractors.

## Data

Driving data was taken across four driving sessions, primarily concentrated on Route 125 in Vermont, and Interstate 93 in Massachusetts, using three devices in a vehicle. One smartphone was mounted in a car, running Physics Toolbox to sample accelerometer data at 100Hz, another smartphone was used running Open GPX Tracker in order to capture real-time GPS coordinates, and a laptop kept track of labels during each drive, with my volunteer operating the labeler.py script in order to mark pothole and normal road surfaces. 

Each session produced three files, an acceleration CSV with ax, ay, az, atotal values mapped to distance elapsed, a CSV with event timetamps, and a GPX file that contained GPS coordinates with timestamps. The pipeline.py script allowed us to merge the three data forms in terms of local time, extracts fixed windows of acceleration data points around each event label, and removes the contaminated events in which potholes and normal labels occur close to each other, in order to avoid cross-class training problems.

Nine dataset configurations were created in this process, with three different window sizes (A: 3.5 seconds, B: 2.5 seconds, C: 1.5 seconds) and three different contamination thresholds (1: 2.5 seconds, 2: 3.0 seconds, 3: 3.5 seconds). In this report, we focus on configuration C3, which is the shortest time window with the most stringent contamination filter, so as to limit normal road data bleediing into pothole windows as well as to limit the chance that 2 labels would be assigned to the same road segment.

The components of data that were used as features were the four acceleration channels from the accelerometer, and the binary label of Normal vs. Pothole is our target. Below, we import the requisite libraries necessary for this notebook, and load the C3 dataset, demonstrate it's structure, and provide a visualization of the geographic distribution of labels via an html map located in the maps directory.

In [17]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
import os
from torch.utils.data import DataLoader
from torchinfo import summary
from src.models import *
from src.helpers import *

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

ACCELS = ['ax', 'ay', 'az', 'atotal']
CLASSES = ['NORMAL', 'POTHOLE']
WINDOW_LENGTHS = {'A': 350, 'B': 250, 'C': 150}
cost_matrix = torch.tensor([[0, 200.0], [400.0, 0]], dtype=torch.float32).to(device)

Using device: mps


In [9]:
df = pd.read_csv('data/intermediary_data/supervised_C3.csv')
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nClass distributions:")
print(df.groupby('event_id')['label'].first().value_counts())
df.head()

Shape: (102169, 13)

Columns: ['event_id', 'label', 'test', 'elapsed_sec', 'wall_clock', 'ax', 'ay', 'az', 'atotal', 'lat', 'lon', 'elevation', 'label_time']

Class distributions:
label
NORMAL     533
POTHOLE    147
Name: count, dtype: int64


,event_id,label,test,elapsed_sec,wall_clock,ax,ay,az,atotal,lat,lon,elevation,label_time
0,test_1_NORMAL_224942647231,NORMAL,test_1,22.901,2026-04-19 22:49:41.901000+00:00,0.94,-0.58,0.95,1.46,44.00799,-73.180151,131.981021,2026-04-19 22:49:42.647231+00:00
1,test_1_NORMAL_224942647231,NORMAL,test_1,22.911,2026-04-19 22:49:41.911000+00:00,0.73,0.06,0.45,0.86,44.00799,-73.180151,131.981021,2026-04-19 22:49:42.647231+00:00
2,test_1_NORMAL_224942647231,NORMAL,test_1,22.921,2026-04-19 22:49:41.921000+00:00,0.77,-0.46,0.63,1.09,44.00799,-73.180151,131.981021,2026-04-19 22:49:42.647231+00:00
3,test_1_NORMAL_224942647231,NORMAL,test_1,22.931,2026-04-19 22:49:41.931000+00:00,1.06,-0.52,0.71,1.38,44.00799,-73.180151,131.981021,2026-04-19 22:49:42.647231+00:00
4,test_1_NORMAL_224942647231,NORMAL,test_1,22.941,2026-04-19 22:49:41.941000+00:00,0.84,-0.03,0.70,1.09,44.00799,-73.180151,131.981021,2026-04-19 22:49:42.647231+00:00


In [ ]:
df = pd.read_csv('data/intermediary_data/supervised_C3.csv')

# find one row per event
events = df.dropna(subset=['lat', 'lon']) # drop no gps rows
events = events.groupby('event_id').first().reset_index() # find first row of each event

m = folium.Map(location=[events['lat'].mean(), events['lon'].mean()], zoom_start=10) # create a map

for _, row in events.iterrows(): # loop through locations
    color = 'red' if row['label'] == 'POTHOLE' else 'blue'
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=6,
        color=color,
        fill=True,
        popup=row['label']
    ).add_to(m)

# places the folium map into the map directory
map_dir = 'maps'
if not os.path.exists(map_dir):
    os.makedirs(map_dir)
    print(f"Created directory: {map_dir}")
map_path = os.path.join(map_dir, 'road_map_C3.html')
m.save(map_path)
print("Map of data has been saved")

results = []


Map of data has been saved


## Our Approach

We used four levels of model complexity on the C3 configuration of our dataset, in increasing level of complexity.

L1: Our baseline, in which we always predict the label of an even to be Normal. This is our naive approach, and sets our bottom line accuracy of roughly 80%, which is high due to the class imbalance, but there is a high false negative rate (zero recall).

L2: A step up, with Logistic Regression on Raw Windows. This is the raw accelerometer window flattened into a feature vector and then passed into the logistic regression model. The threshold is tuned on the validation set to make a decision boundary which minimizes the expected cost, and the loss funciton is binary cross-entropy. The optimizer used is Adam.

L3: A similar approach to L2, with Logistic Regression on Engineered Features. Here we use summary statistics (mean, std, max, min) on each of the four channels to create a 16-dimensional feature vector. This allows us to better capture the acceleration peaks and changes that are representative of a pothole. Once agiain, the threshold is tuned on the validation set to make a decision boundary which minimizes the expected cost, and the loss funciton is binary cross-entropy. However, this case gets better training on the shape of the signal due to the summary statistics. The optimizer used is Adam.

L4: A 1D Convolutional Neural Network. A four layer 1D CNN that learns the features themselves from raw accelerometer signals within each labeled window. We use Adam as our optimizer and BCE as our loss function, and use checkpointing on the best F1 score (rather than accuracy, which favors balanced classes) over 100 epochs. We then implement some random time-shift to augment our data, wrapping data around within the same window. The model was trained using train_cnn.py, and the checkpoint we found is located in the root.

All models use 80/20 train/validation data split in order to prevent leackage of data, and we make evaluations using accuracy, F1, and expected cost primarily, and recall and precision less explicitly, using a cost matrix to penalize missed potholes (FN, 400) twice as heavily as false alarms (FP, 200). TN and TP were left as 0 cost, as they are correct classifications. We know that the cost of hitting a pothole that is undetected is about twice as large as sending an individual to a site to conduct a repair that isn't necessary, which is the reasoning for the costs assigned above.


In [11]:
torch.manual_seed(42)
np.random.seed(42)

y_rows = [CLASSES.index(group['label'].iloc[0]) for _, group in df.groupby('event_id')]
y = torch.tensor(y_rows, dtype=torch.float32).to(device)

idx = torch.randperm(len(y))
split = int(0.8 * len(y))
y_train, y_val = y[idx[:split]], y[idx[split:]]

print("Baseline (always predict NORMAL)\n")
for split_name, y_true in [("Train", y_train), ("Val", y_val)]:
    y_pred = torch.zeros(len(y_true)).to(device)
    cm = confusion_matrix(y_pred, y_true)
    p  = precision(cm)
    r  = recall(cm)
    f1 = f1_score(p, r)
    acc = accuracy(y_pred, y_true)
    ec = expected_cost(cm, cost_matrix)
    print(f"  {split_name} Confusion Matrix:\n{cm.cpu()}")
    print(f"  Accuracy:      {acc:.3f}")
    print(f"  Expected Cost: {ec:.3f}")
    print(f"  Precision:     {p:.3f}")
    print(f"  Recall:        {r:.3f}")
    print(f"  F1:            {f1:.3f}\n")
results.append(("L1: Baseline", f1, acc, ec, cm))


Baseline (always predict NORMAL)

  Train Confusion Matrix:
tensor([[433.,   0.],
        [111.,   0.]])
  Accuracy:      0.796
  Expected Cost: 81.618
  Precision:     0.000
  Recall:        0.000
  F1:            0.000

  Val Confusion Matrix:
tensor([[100.,   0.],
        [ 36.,   0.]])
  Accuracy:      0.735
  Expected Cost: 105.882
  Precision:     0.000
  Recall:        0.000
  F1:            0.000



In [12]:
torch.manual_seed(42)
np.random.seed(42)

T = torch.linspace(0, 1, 101).to(device)
window_len = WINDOW_LENGTHS['C']

levels = {
    "L2: Logistic Regression (raw window)": flatten_window,
    "L3: Logistic Regression (engineered features)": engineer_features,
}

# extract labels
y_rows = [CLASSES.index(group['label'].iloc[0]) for _, group in df.groupby('event_id')]
y = torch.tensor(y_rows, dtype=torch.float32).to(device)

# train/val split 80/20
idx = torch.randperm(len(y))
split_vals = int(0.8 * len(y))

for level_name, feature_fn in levels.items():

    # feature extraction
    X_rows = []
    for event, group in df.groupby('event_id'):
        samples = group[ACCELS][:window_len].values
        X_rows.append(feature_fn(samples))

    X = torch.tensor(np.array(X_rows), dtype=torch.float32).to(device)

    # normalize X
    X = (X - X.mean(dim=0)) / (X.std(dim=0) + 1e-8)

    X_training, X_validation = X[idx[:split_vals]], X[idx[split_vals:]]
    y_training, y_validation = y[idx[:split_vals]].reshape(-1, 1), y[idx[split_vals:]].reshape(-1, 1)

    # training loop
    model = LogisticModel(X_training.shape[1], device)
    opt = torch.optim.Adam([model.w], lr=1e-3)

    for epoch in range(1000):
        q = model.forward(X_training)
        loss = binary_cross_entropy(q, y_training)
        opt.zero_grad()
        loss.backward()
        opt.step()

    # get probabilities for validation set
    q_validation = model.forward(X_validation).detach()

    # compute costs and accuracies over threshold values, adapted from lecture
    costs, accuracies = costs_and_accuracies(q_validation, y_validation, cost_matrix, T)
    best_t_cost = T[torch.argmin(torch.tensor(costs))].item()
    best_t_acc = T[torch.argmax(torch.tensor(accuracies))].item()

    print(f"\n{level_name} — C3")
    print(f"  best expected cost = {min(costs):.3f} at t={best_t_cost:.2f}")
    print(f"  best accuracy      = {max(accuracies):.3f} at t={best_t_acc:.2f}")

    # evaluate at best threshold
    for split_name, X_s, y_s in [("Train", X_training, y_training), ("Val", X_validation, y_validation)]:
        y_pred = (model.forward(X_s) >= best_t_cost).float()
        cm = confusion_matrix(y_pred, y_s)
        p  = precision(cm)
        r  = recall(cm)
        f1 = f1_score(p, r)

        acc = accuracy(y_pred, y_s)
        ec = expected_cost(cm, cost_matrix)

        print(f"  {split_name} Confusion Matrix:\n{cm.cpu()}")
        print(f"  Accuracy:      {acc:.3f}")
        print(f"  Expected Cost: {ec:.3f}")
        print(f"  Precision:     {p:.3f}")
        print(f"  Recall:        {r:.3f}")
        print(f"  F1:            {f1:.3f}\n")

    results.append((level_name, f1, acc, ec, cm))


L2: Logistic Regression (raw window) — C3
  best expected cost = 39.706 at t=0.97
  best accuracy      = 0.882 at t=0.99
  Train Confusion Matrix:
tensor([[433.,   0.],
        [ 12.,  99.]])
  Accuracy:      0.978
  Expected Cost: 8.824
  Precision:     1.000
  Recall:        0.892
  F1:            0.943

  Val Confusion Matrix:
tensor([[91.,  9.],
        [ 9., 27.]])
  Accuracy:      0.868
  Expected Cost: 39.706
  Precision:     0.750
  Recall:        0.750
  F1:            0.750


L3: Logistic Regression (engineered features) — C3
  best expected cost = 22.059 at t=0.80
  best accuracy      = 0.919 at t=0.80
  Train Confusion Matrix:
tensor([[423.,  10.],
        [ 12.,  99.]])
  Accuracy:      0.960
  Expected Cost: 12.500
  Precision:     0.908
  Recall:        0.892
  F1:            0.900

  Val Confusion Matrix:
tensor([[93.,  7.],
        [ 4., 32.]])
  Accuracy:      0.919
  Expected Cost: 22.059
  Precision:     0.821
  Recall:        0.889
  F1:            0.853



In [13]:
window_length = WINDOW_LENGTHS['C']

torch.manual_seed(42)
np.random.seed(42)

# split events
event_ids = df['event_id'].unique()
idx = torch.randperm(len(event_ids))
split = int(0.8 * len(idx))

train_ids = event_ids[idx[:split].numpy()]
val_ids   = event_ids[idx[split:].numpy()]

# create dataloaders, using dataset objects like miniproject
train_loader = DataLoader(RoadSurfaceDataset(df[df['event_id'].isin(train_ids)], window_length, device, augment=True),
                          batch_size=16, shuffle=True)
val_loader   = DataLoader(RoadSurfaceDataset(df[df['event_id'].isin(val_ids)], window_length, device, augment=False),
                          batch_size=16, shuffle=False)

# load best saved model instead of retraining
model = RoadSurfaceCNN().to(device)
model.load_state_dict(torch.load('best_model_C3.pt', map_location=device))

print("CNN — C3")
print(summary(model, input_size=next(iter(train_loader))[0].shape, device=device))

# final metrics on train and val sets
for split_name, loader in [("Train", train_loader), ("Val", val_loader)]:
    acc, ec, f1, cm = evaluate(model, loader, cost_matrix)
    p = precision(cm)
    r = recall(cm)
    print(f"\n  {split_name} Confusion Matrix:\n{cm.cpu()}")
    print(f"  Accuracy:      {acc:.3f}")
    print(f"  Expected Cost: {ec:.3f}")
    print(f"  Precision:     {p:.3f}")
    print(f"  Recall:        {r:.3f}")
    print(f"  F1:            {f1:.3f}")

results.append(("L4: 1D CNN", f1, acc, ec, cm))


CNN — C3
Layer (type:depth-idx)                   Output Shape              Param #
RoadSurfaceCNN                           [16, 1]                   --
├─Sequential: 1-1                        [16, 1]                   --
│    └─Conv1d: 2-1                       [16, 16, 150]             208
│    └─ReLU: 2-2                         [16, 16, 150]             --
│    └─MaxPool1d: 2-3                    [16, 16, 75]              --
│    └─Conv1d: 2-4                       [16, 32, 75]              1,568
│    └─ReLU: 2-5                         [16, 32, 75]              --
│    └─MaxPool1d: 2-6                    [16, 32, 37]              --
│    └─Conv1d: 2-7                       [16, 64, 37]              6,208
│    └─ReLU: 2-8                         [16, 64, 37]              --
│    └─MaxPool1d: 2-9                    [16, 64, 18]              --
│    └─Conv1d: 2-10                      [16, 128, 18]             24,704
│    └─ReLU: 2-11                        [16, 128, 18]           

In [14]:
for model_name, f1, acc, cost, cm in results:
    print(f"Model name:{model_name:<30} \n| F1 Score:{f1:.3f}  | Accuracy:{acc:.3f}  | Expected Cost{cost:.3f}")
    print(f"Confusion Matrix: \n{cm}")

Model name:L1: Baseline                   
| F1 Score:0.000  | Accuracy:0.735  | Expected Cost105.882
Confusion Matrix: 
tensor([[100.,   0.],
        [ 36.,   0.]], device='mps:0')
Model name:L2: Logistic Regression (raw window) 
| F1 Score:0.750  | Accuracy:0.868  | Expected Cost39.706
Confusion Matrix: 
tensor([[91.,  9.],
        [ 9., 27.]], device='mps:0')
Model name:L3: Logistic Regression (engineered features) 
| F1 Score:0.853  | Accuracy:0.919  | Expected Cost22.059
Confusion Matrix: 
tensor([[93.,  7.],
        [ 4., 32.]], device='mps:0')
Model name:L4: 1D CNN                     
| F1 Score:0.941  | Accuracy:0.971  | Expected Cost5.882
Confusion Matrix: 
tensor([[100.,   4.],
        [  0.,  32.]], device='mps:0')


## Results

The results show a definite improvement and gradual progress among the four model levels. With the base model, our F1 score is the worst it could be, but achieves baseline accuracy, and has high expected cost, especially due to the many potholes missed, which costs a lot when examining our cost matrix. The L2 overfits fairly badly, with inferior validation performance compared to our training performance, and has relatively low F1 score and minimal accuracy improvement compared to the base rate of ~70-80%, presumably due to the flattened window being difficult for our model to learn the signal. L3 is a substantial improvement, as by reducing the amount of features to 16, which also capture a great deal of the “peaking” nature of acceleration during a pothole event. The 1D CNN has the best performance, achieving minimal expected costs, high accuracy, and high F1 score, showing that learning features from the raw signal has a high upside.

## Concluding Discussion

Our project had high degrees of success. It demonstrated that by using smartphone accelerometer data, we can reliably classify road events. The 1D CNN achieved extremely high F1 scores on configuration C3, which confirms that the learned features were able to outperform our raw data and hand-engineered features in our more basic logistic regression model. High degrees of accuracy were also achieved in both the hand-engineered logistic regression model and in the 1D CNN, which seems to be consistent with previous work.

## Contributions

This was an individual project. I researched adequate methods of capturing GPS and Acceleration Data, and created a labeler script (labeler.py) in order to mark sections of road off as potholes or normal sections. I conducted the test drives to use the labeler script, but I required the help of a volunteer in order to press keys corresponding with labels on my laptop while I drove. I then wrote a data pipeline (pipeline.py) in order to clean up the data collection windows to create centralized spreadsheets that reflect the three sources of data in one place. I then made a visualizer (plots.py) and ran a baseline test (a model that picks the most common class), and two levels (engineering and non-engineered features ) of a logistic regression model, in order to gauge which data configuration best demonstrated the prediction gains of each successively "better" model. Then, I trained three models on our best data configuration, C3, and developed our comparisons, map visualization, "blog post" writing, and this final report.